swin transformer


In [8]:
# ======================================================
# EfficientFormer L3 - Full Fine-Tune (Streamlit-ready single .pkl)
# ======================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from timm import create_model
import joblib
from tqdm import tqdm
import pickle

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"
SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "efficientformer_full_finetune_streamlit.pkl")

# ======================================================
# CONFIG
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 30
PATIENCE = 5
LR = 3e-5
WEIGHT_DECAY = 0.05

# ======================================================
# DATA TRANSFORMS (ImageNet stats)
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(os.path.join(BASE_DIR, "train"), transform=transform)
val_ds   = datasets.ImageFolder(os.path.join(BASE_DIR, "test"), transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

class_names = train_ds.classes
print("Classes:", class_names)

# ======================================================
# MODEL: EfficientFormer L3 - FULL FINE-TUNE
# ======================================================
model = create_model("efficientformer_l3", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Full fine-tune: semua parameter dilatih
for param in model.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler()  # Mixed precision

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("Starting FULL fine-tune EfficientFormer L3...")
for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Record metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # ---------------- SAVE BEST (.pkl Streamlit-ready) ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        # Artifact: gabungkan model + metrics
        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "efficientformer_l3 (Full Fine-Tune)",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {"mean": [0.485,0.456,0.406], "std": [0.229,0.224,0.225]},
            "best_val_acc": best_val_acc,
            "epoch": epoch + 1,
            "metrics": {
                "train_losses": train_losses,
                "val_losses": val_losses,
                "train_accs": train_accs,
                "val_accs": val_accs
            }
        }

        # Simpan satu file saja untuk Streamlit
        joblib.dump(artifact, MODEL_PATH)
        print(f">>> BEST MODEL + METRICS SAVED! Val Acc: {best_val_acc:.4f} -> {MODEL_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping triggered!")
            break

print("\n[SUCCESS] Full Fine-Tune EfficientFormer L3 selesai!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Model + Metrics .pkl siap untuk Streamlit: {MODEL_PATH}")


DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


C:\Users\acer\AppData\Local\Temp\ipykernel_39412\1948646540.py:74: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Mixed precision


Starting FULL fine-tune EfficientFormer L3...


Epoch 1/30 [Train]:   0%|          | 0/184 [00:00<?, ?it/s]C:\Users\acer\AppData\Local\Temp\ipykernel_39412\1948646540.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Train]: 100%|██████████| 184/184 [04:43<00:00,  1.54s/it]
C:\Users\acer\AppData\Local\Temp\ipykernel_39412\1948646540.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 1] Train Acc: 0.6844 | Val Acc: 0.7088 | Train Loss: 0.7291 | Val Loss: 0.7408
>>> BEST MODEL + METRICS SAVED! Val Acc: 0.7088 -> D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\efficientformer_full_finetune_streamlit.pkl


...

[Epoch 15] Train Acc: 1.0000 | Val Acc: 0.8956 | Train Loss: 0.0002 | Val Loss: 0.4722


Epoch 16/30 [Train]: 100%|██████████| 184/184 [04:55<00:00,  1.61s/it]


[Epoch 16] Train Acc: 1.0000 | Val Acc: 0.9002 | Train Loss: 0.0002 | Val Loss: 0.4713


Epoch 17/30 [Train]: 100%|██████████| 184/184 [04:55<00:00,  1.60s/it]


[Epoch 17] Train Acc: 1.0000 | Val Acc: 0.8976 | Train Loss: 0.0001 | Val Loss: 0.4821
>>> Early stopping triggered!

[SUCCESS] Full Fine-Tune EfficientFormer L3 selesai!
Best Validation Accuracy: 0.9012
Model + Metrics .pkl siap untuk Streamlit: D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\efficientformer_full_finetune_streamlit.pkl


EfficientFormer

In [ ]:
# ======================================================
# EfficientFormer L3 - Full Fine-Tune (Streamlit-ready single .pkl)
# ======================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from timm import create_model
import joblib
from tqdm import tqdm

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"
SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "efficientformer_full_finetune_streamlit.pkl")

# ======================================================
# CONFIG
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 30
PATIENCE = 5
LR = 3e-5
WEIGHT_DECAY = 0.05

# ======================================================
# DATA TRANSFORMS
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(os.path.join(BASE_DIR, "train"), transform=transform)
val_ds   = datasets.ImageFolder(os.path.join(BASE_DIR, "test"), transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

class_names = train_ds.classes
print("Classes:", class_names)

# ======================================================
# MODEL: EfficientFormer L3 - FULL FINE-TUNE
# ======================================================
model = create_model("efficientformer_l3", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Full fine-tune
for param in model.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler()  # Mixed precision

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("Starting FULL fine-tune EfficientFormer L3...")
for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Record metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # ---------------- SAVE BEST (.pkl Streamlit-ready) ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "efficientformer_l3 (Full Fine-Tune)",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            },
            "best_val_acc": best_val_acc,
            "epoch": epoch + 1,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> BEST MODEL & METRICS SAVED! Val Acc: {best_val_acc:.4f} -> {MODEL_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping triggered!")
            break

print("\n[SUCCESS] Full Fine-Tune EfficientFormer L3 selesai!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Model & Metrics .pkl siap untuk Streamlit: {MODEL_PATH}")

DEVICE: cuda
Classes: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3290777499.py:73: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Mixed precision


Starting FULL fine-tune EfficientFormer L3...


Epoch 1/30 [Train]:   0%|          | 0/184 [00:00<?, ?it/s]C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3290777499.py:96: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Train]: 100%|██████████| 184/184 [03:41<00:00,  1.21s/it]
C:\Users\acer\AppData\Local\Temp\ipykernel_39412\3290777499.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Epoch 1] Train Acc: 0.6711 | Val Acc: 0.7508 | Train Loss: 0.7436 | Val Loss: 0.5448
>>> BEST MODEL & METRICS SAVED! Val Acc: 0.7508 -> D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model\efficientformer_full_finetune_streamlit.pkl


...

Epoch 12/30 [Train]: 100%|██████████| 184/184 [04:41<00:00,  1.53s/it]


[Epoch 12] Train Acc: 1.0000 | Val Acc: 0.8930 | Train Loss: 0.0003 | Val Loss: 0.4044


Epoch 13/30 [Train]: 100%|██████████| 184/184 [04:41<00:00,  1.53s/it]


[Epoch 13] Train Acc: 1.0000 | Val Acc: 0.8900 | Train Loss: 0.0002 | Val Loss: 0.4243


Epoch 14/30 [Train]:  66%|██████▋   | 122/184 [03:08<01:30,  1.46s/it]

ViT B/16

In [ ]:
# ======================================================
# ViT Base Patch16/224 - Fine-Tune HEAD ONLY (Streamlit-ready single .pkl)
# ======================================================

import os
import joblib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
import timm

# ======================================================
# DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ======================================================
# PATH CONFIG
# ======================================================
BASE_DIR = r"D:\alzheimer detection.v1i.folder\processed_dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")  # ganti sesuai dataset

SAVE_DIR = r"D:\alzheimer detection.v1i.folder\Dashboard\src\Transformer\model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "vit_base_finetune_head_streamlit.pkl")

# ======================================================
# CONFIG
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
NUM_WORKERS = 2
EPOCHS = 20
PATIENCE = 3
LR = 1e-4

# ======================================================
# TRANSFORM
# ======================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================================================
# DATASET & DATALOADER
# ======================================================
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

class_names = train_dataset.classes
print("Classes:", class_names)

# ======================================================
# MODEL: ViT Base Patch16/224 (Head only fine-tune)
# ======================================================
model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Freeze backbone, hanya latih head
for name, param in model.named_parameters():
    param.requires_grad = "head" in name  # classifier bernama 'head'

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scaler = torch.cuda.amp.GradScaler()  # Mixed precision

# ======================================================
# TRAINING LOOP
# ======================================================
best_val_acc = 0.0
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("Starting training (head only)...")
for epoch in range(EPOCHS):
    # ---------------- TRAIN ----------------
    model.train()
    running_loss = correct = total = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = val_correct = val_total = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Record
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # ---------------- SAVE BEST (.pkl Streamlit-ready) ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        artifact = {
            "model_state_dict": model.state_dict(),
            "architecture": "vit_base_patch16_224 (Head Only Fine-Tune)",
            "num_classes": NUM_CLASSES,
            "class_names": class_names,
            "img_size": IMG_SIZE,
            "normalization": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            },
            "best_val_acc": best_val_acc,
            "epoch": epoch + 1,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs
        }

        joblib.dump(artifact, MODEL_PATH)
        print(f">>> BEST MODEL & METRICS SAVED! Val Acc: {best_val_acc:.4f} → {MODEL_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(">>> Early stopping triggered!")
            break

print("\n[SUCCESS] ViT Base Patch16/224 (head fine-tune) selesai!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Model & Metrics .pkl siap untuk Streamlit: {MODEL_PATH}")
